# Prediksi Navigasi Wikipedia (Wikispeedia)

Notebook ini membangun graf hyperlink Wikipedia dari screenshot halaman menggunakan OCR, lalu memprediksi artikel berikutnya yang diklik navigator dengan mencari jalur terpendek di graf tersebut menuju artikel target. Pendekatan awal memakai machine learning, LightGBM di atas fitur similarity teks dan kategori, tapi hasilnya cuma sekitar 0.28 karena model kesulitan belajar pola kedekatan dari data train yang hanya punya 360 target artikel unik. Navigasi Wikipedia pada dasarnya adalah usaha mencari jalur terpendek menuju artikel populer atau hub yang relevan, jadi pendekatan graf langsung jauh lebih pas dibanding model yang harus belajar pola itu dari contoh terbatas.

Dependensi di luar bawaan Kaggle adalah easyocr, rapidfuzz, dan networkx. Bagian paling lama di notebook ini adalah OCR karena harus membaca hyperlink dari ribuan screenshot, sisanya (bangun graf, BFS, prediksi) selesai dalam hitungan detik.

## Konfigurasi dan dependensi

In [ ]:
import os, re, json, time, sys, subprocess, warnings
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None          # screenshot bisa setinggi ~39 ribu px
np.random.seed(42)


class CFG:
    # --- path (Kaggle) ---
    DATA   = "/kaggle/input/competitions/datathon-task-2/dataset-task2"
    WORK   = "/kaggle/working"
    SHOTS  = DATA + "/screenshots"
    OCR_CACHE = WORK + "/ocr_links_full.json"
    SUB_PATH  = WORK + "/submission.csv"

    # --- OCR ---
    OCR_LIMIT      = None    # isi misal 50 untuk smoke test cepat, None = semua artikel
    LINK_GAP_PX    = 20      # dilasi horizontal (px), menyatukan kata dalam satu link
                              # tanpa menyatukan ke link lain yang terpisah
    MIN_LINE_H     = 8       # px, menyaring noise kecil di mask biru
    MAX_LINE_H     = 70      # px, menyaring blob yang tergabung tidak wajar
    MIN_LINE_W     = 15      # px, menyaring noise kecil di mask biru
    LINE_SPACER    = 6       # px jarak antar span yang ditumpuk dalam satu crop halaman
    MAX_SPANS_PER_PAGE = 400 # batas aman untuk halaman dengan link sangat padat
    BATCH_HEIGHT_PX = 12000  # target tinggi komposit per panggilan OCR, sekitar 15-20 halaman per batch
    PAGE_GAP_PX     = 40     # px jarak antar halaman berbeda dalam satu batch
    OCR_CONF_MIN   = 0.30
    MIN_BLUE_PIX   = 60      # halaman dengan piksel biru terlalu sedikit dianggap tidak punya link
    FUZZY_CUTOFF   = 86      # skor rapidfuzz minimum untuk menerima hasil OCR sebagai judul artikel

    # --- skor graf ---
    PAGERANK_ALPHA   = 0.85  # damping factor standar PageRank
    UNREACHABLE_DIST = 999   # jarak sentinel kalau kandidat tidak bisa mencapai target di graf


def _ensure(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)


_ensure("easyocr")
_ensure("rapidfuzz")
_ensure("networkx")

import easyocr
from rapidfuzz import process, fuzz
import networkx as nx

## Baca data

Ada lima file, articles berisi judul tiap artikel, categories berisi kategori tiap artikel yang bisa lebih dari satu, states_train dan states_test berisi baris current, target, dan (khusus train) next yang jadi label, serta folder screenshots berisi tangkapan layar tiap halaman. Semua tabel terhubung lewat article_id.

In [ ]:
articles   = pd.read_csv(f"{CFG.DATA}/articles.csv")
categories = pd.read_csv(f"{CFG.DATA}/categories.csv")
train      = pd.read_csv(f"{CFG.DATA}/states_train.csv")
test       = pd.read_csv(f"{CFG.DATA}/states_test.csv")
sample_sub = pd.read_csv(f"{CFG.DATA}/sample_submission.csv")

N = int(articles.article_id.max()) + 1
for name, df in [("articles", articles), ("categories", categories),
                 ("train", train), ("test", test), ("sample_sub", sample_sub)]:
    print(f"{name:11s} {df.shape}")
print("N articles:", N)

## Eksplorasi singkat

Sebelum masuk ke OCR, ada dua hal yang perlu dicek karena langsung memengaruhi desain solusi. Pertama, apakah target artikel di data train tersebar merata atau timpang. Kedua, apakah artikel yang jadi jawaban next didominasi segelintir artikel populer saja.

In [ ]:
tgt_counts = train.target_article_id.value_counts()
nxt_counts = train.next_article_id.value_counts()
print("Jumlah target unik di train:", train.target_article_id.nunique())
print("Setiap target muncul persis segini kali (min, max):", tgt_counts.min(), tgt_counts.max())
print()
print("Jumlah next unik di train:", train.next_article_id.nunique())
print("Porsi baris yang next-nya adalah artikel paling populer:",
      f"{nxt_counts.iloc[0] / len(train) * 100:.2f}%")
print("10 artikel next paling sering muncul (judul, jumlah):")
top_next = nxt_counts.head(10)
titles_map = articles.set_index("article_id").title
for aid, cnt in top_next.items():
    print(f"  {titles_map[aid]:30s} {cnt}")

Target artikel di train ternyata tersebar rata, tiap target muncul tepat 25 kali, jadi tidak ada bias target tertentu yang lebih sering. Sebaliknya, next sangat timpang, didominasi artikel besar seperti United States, Europe, dan United Kingdom. Ini konsisten dengan cara orang bernavigasi di Wikipedia, mereka cenderung menuju artikel hub yang punya banyak koneksi dulu sebelum mengarah spesifik ke target. Temuan ini yang mendasari kenapa skor hub (in-degree dan PageRank) dipakai sebagai penentu utama saat ada beberapa kandidat link dengan jarak yang sama ke target.

In [ ]:
cat_per_article = categories.groupby("article_id").size()
print("Rata-rata kategori per artikel:", round(cat_per_article.mean(), 2))
print("Artikel tanpa kategori sama sekali:", N - cat_per_article.shape[0])
subj = categories.category.str.split(".").str[1]
print("Lima subjek kategori paling umum:")
print(subj.value_counts().head(5).to_string())

Rata-rata artikel cuma punya sekitar satu kategori, dan sebagian kecil artikel sama sekali tidak punya kategori. Karena sinyalnya tipis dan tidak semua artikel tercakup, kategori tidak dipakai sebagai sinyal utama di notebook ini, fokus tetap di struktur graf hyperlink yang jauh lebih lengkap dan langsung merepresentasikan pilihan link yang benar-benar tersedia di tiap halaman.

## Siapkan lookup judul

OCR akan mengembalikan potongan teks, teks itu perlu dipetakan balik ke article_id lewat judul artikel. Judul dinormalisasi dulu (huruf kecil, tanda baca dibuang) supaya pencocokan tidak terlalu sensitif terhadap kapitalisasi atau tanda baca kecil.

In [ ]:
titles = np.array([""] * N, dtype=object)
for i, t in zip(articles.article_id, articles.title):
    titles[i] = str(t)


def normalize(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()


title_norms = [normalize(titles[i]) for i in range(N)]
exact_lookup = {}
for i, n in enumerate(title_norms):
    if n and n not in exact_lookup:
        exact_lookup[n] = i

## OCR, membaca hyperlink dari screenshot

Hyperlink di screenshot Wikipedia selalu berwarna biru, jadi langkah pertama adalah memisahkan piksel biru dari sisa halaman. Piksel biru yang berdekatan digabung jadi satu span lewat connected component, satu span kira-kira mewakili satu link, termasuk link yang terdiri dari beberapa kata. Tiap span dipotong ketat dan ditumpuk jadi satu gambar komposit per halaman, sehingga OCR hanya membaca teks link yang relevan, bukan seluruh isi artikel.

Percobaan awal memanggil OCR satu kali per halaman, hasilnya lambat sekali karena ternyata biaya tiap panggilan OCR didominasi overhead tetap, bukan ukuran gambar. Solusinya, banyak halaman digabung jadi satu komposit besar per panggilan OCR (sekitar 15 sampai 20 halaman sekaligus), sehingga jumlah panggilan turun drastis. Hasil OCR dari satu komposit besar kemudian dipetakan balik ke halaman dan span asalnya berdasarkan posisi vertikal, lalu tiap span dicocokkan ke judul artikel lewat pencocokan teks persis dulu, baru fuzzy matching kalau tidak ketemu persis. Hasilnya disimpan ke cache di disk supaya tidak perlu diulang.

In [ ]:
def blue_mask(a: np.ndarray) -> np.ndarray:
    R = a[:, :, 0].astype(np.int16)
    G = a[:, :, 1].astype(np.int16)
    B = a[:, :, 2].astype(np.int16)
    return (B > 110) & (B - R > 55) & (B - G > 30) & (R < 130)


def match_text_to_id(text: str):
    n = normalize(text)
    if not n:
        return None
    if n in exact_lookup:
        return exact_lookup[n]
    hit = process.extractOne(n, title_norms, scorer=fuzz.ratio, score_cutoff=CFG.FUZZY_CUTOFF)
    return hit[2] if hit else None


def _blue_spans(mask: np.ndarray) -> list:
    dilated = ndimage.binary_dilation(mask, structure=np.ones((3, CFG.LINK_GAP_PX), bool))
    labeled, n = ndimage.label(dilated)
    if n == 0:
        return []
    spans = []
    for sl in ndimage.find_objects(labeled):
        if sl is None:
            continue
        h = sl[0].stop - sl[0].start
        w = sl[1].stop - sl[1].start
        if CFG.MIN_LINE_H <= h <= CFG.MAX_LINE_H and w >= CFG.MIN_LINE_W:
            spans.append(sl)
    spans.sort(key=lambda s: s[0].start)
    return spans[:CFG.MAX_SPANS_PER_PAGE]


def _build_page_composite(aid: int):
    path = f"{CFG.SHOTS}/{aid}.png"
    try:
        img = np.asarray(Image.open(path).convert("RGB"))
    except Exception:
        return None, None
    H, W = img.shape[0], img.shape[1]
    mask = blue_mask(img)
    if int(mask.sum()) < CFG.MIN_BLUE_PIX:
        return None, None

    spans = _blue_spans(mask)
    if not spans:
        return None, None

    rows, row_ranges, cursor = [], [], 0
    for sl in spans:
        r0, r1 = max(sl[0].start - 3, 0), min(sl[0].stop + 3, H)
        c0, c1 = max(sl[1].start - 3, 0), min(sl[1].stop + 3, W)
        crop = img[r0:r1, c0:c1]
        line_mask = blue_mask(crop)
        canvas_row = np.full((crop.shape[0], W, 3), 255, np.uint8)
        canvas_row[:, :crop.shape[1]][line_mask] = 0
        rows.append(canvas_row)
        rows.append(np.full((CFG.LINE_SPACER, W, 3), 255, np.uint8))
        h = crop.shape[0]
        row_ranges.append((cursor, cursor + h, (r0 + r1) / 2.0 / max(H, 1)))
        cursor += h + CFG.LINE_SPACER
    return rows, row_ranges


def _decode_batch_results(results, page_starts) -> dict:
    starts_arr = np.array([p[0] for p in page_starts])
    buckets = {i: {} for i in range(len(page_starts))}
    for bbox, text, conf in results:
        if conf < CFG.OCR_CONF_MIN:
            continue
        yc = (bbox[0][1] + bbox[2][1]) / 2.0
        pidx = int(np.searchsorted(starts_arr, yc, side="right") - 1)
        if pidx < 0 or pidx >= len(page_starts):
            continue
        pstart, pend, aid, row_ranges = page_starts[pidx]
        if not (pstart <= yc <= pend):
            continue
        local_y = yc - pstart
        s_starts = np.array([r[0] for r in row_ranges])
        sidx = int(np.searchsorted(s_starts, local_y, side="right") - 1)
        if sidx < 0 or sidx >= len(row_ranges) or not (row_ranges[sidx][0] <= local_y <= row_ranges[sidx][1]):
            continue
        xc = (bbox[0][0] + bbox[1][0]) / 2.0
        buckets[pidx].setdefault(sidx, []).append((xc, text))

    page_out = {}
    for pidx, (pstart, pend, aid, row_ranges) in enumerate(page_starts):
        found = {}
        for sidx, words in buckets[pidx].items():
            words.sort(key=lambda t: t[0])
            phrase = " ".join(w for _, w in words)
            cid = match_text_to_id(phrase)
            if cid is None or cid == aid:
                continue
            y_frac = row_ranges[sidx][2]
            if cid not in found or y_frac < found[cid]:
                found[cid] = y_frac
        page_out[aid] = [[int(k), round(v, 4)] for k, v in found.items()]
    return page_out


def run_ocr_all() -> dict:
    need = list(range(N)) if not CFG.OCR_LIMIT else list(range(min(N, CFG.OCR_LIMIT)))

    if os.path.exists(CFG.OCR_CACHE):
        with open(CFG.OCR_CACHE) as f:
            cached = {int(k): v for k, v in json.load(f).items()}
        if all(a in cached for a in need):
            print(f"Loaded OCR cache: {CFG.OCR_CACHE} ({len(cached)} pages)")
            return cached
        print(f"Cache incomplete ({len(cached)} cached, {len(need)} needed), rebuilding")

    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"CUDA available: {cuda_ok}")
    if cuda_ok:
        print(f"GPU: {torch.cuda.get_device_name(0)} (x{torch.cuda.device_count()} visible)")
    else:
        print("WARNING: no GPU detected, OCR will run on CPU and be much slower.")
    reader = easyocr.Reader(["en"], gpu=cuda_ok)

    out: dict = {}
    t0 = time.time()
    batch_pages, batch_rows, batch_h = [], [], 0

    def flush_batch():
        nonlocal batch_pages, batch_rows, batch_h
        if not batch_pages:
            return
        composite = np.vstack(batch_rows)
        try:
            results = reader.readtext(composite, detail=1, paragraph=False,
                                      canvas_size=composite.shape[0] + 100)
        except TypeError:
            try:
                results = reader.readtext(composite, detail=1, paragraph=False)
            except Exception:
                results = []
        except Exception:
            results = []
        page_starts, cursor = [], 0
        for aid, rows, row_ranges in batch_pages:
            h = sum(r.shape[0] for r in rows)
            page_starts.append((cursor, cursor + h, aid, row_ranges))
            cursor += h + CFG.PAGE_GAP_PX
        out.update(_decode_batch_results(results, page_starts))
        batch_pages, batch_rows, batch_h = [], [], 0

    for j, aid in enumerate(need):
        rows, row_ranges = _build_page_composite(aid)
        if rows is None:
            out[aid] = []
        else:
            page_h = sum(r.shape[0] for r in rows)
            if batch_pages and batch_h + page_h > CFG.BATCH_HEIGHT_PX:
                flush_batch()
            batch_pages.append((aid, rows, row_ranges))
            batch_rows.extend(rows)
            batch_rows.append(np.full((CFG.PAGE_GAP_PX, rows[0].shape[1], 3), 255, np.uint8))
            batch_h += page_h + CFG.PAGE_GAP_PX
        if (j + 1) % 200 == 0:
            flush_batch()
            avg = np.mean([len(v) for v in out.values()]) if out else 0.0
            print(f"  OCR {j + 1}/{len(need)} pages | {time.time() - t0:.0f}s | avg links/page={avg:.1f}")
    flush_batch()

    with open(CFG.OCR_CACHE, "w") as f:
        json.dump({str(k): v for k, v in out.items()}, f)
    print(f"OCR done: {len(out)} pages in {time.time() - t0:.0f}s, cached")
    return out


OCR_LINKS = run_ocr_all()
_pages_with_links = sum(len(v) > 0 for v in OCR_LINKS.values())
print(f"Pages with >=1 extracted link: {_pages_with_links}/{len(OCR_LINKS)}")

## Bangun graf hyperlink

Setiap link hasil OCR jadi satu edge berarah dari halaman asal ke halaman tujuan. Dari graf ini dihitung dua skor struktural, in-degree yaitu berapa banyak halaman lain yang mengarah ke suatu artikel, dan PageRank yang mengukur seberapa sentral posisi artikel itu di jaringan. Keduanya dikalikan jadi satu skor hub. Skor ini murni berdasarkan struktur graf, tidak memakai informasi next_article_id sama sekali, jadi aman dipakai walau target di data test sama sekali berbeda dari target di data train.

In [ ]:
G = nx.DiGraph()
G.add_nodes_from(range(N))
for aid, links in OCR_LINKS.items():
    for cid, _ in links:
        if cid != aid:
            G.add_edge(int(aid), int(cid))
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

IN_DEGREE = np.zeros(N, np.float64)
for node, d in G.in_degree():
    IN_DEGREE[node] = d

_pagerank = nx.pagerank(G, alpha=CFG.PAGERANK_ALPHA)
PAGERANK = np.array([_pagerank.get(i, 0.0) for i in range(N)], np.float64)

HUB_SCORE = IN_DEGREE * PAGERANK

G_REV = G.reverse(copy=False)

Graf yang terbentuk punya 4604 node dan sekitar 86 ribu edge, jadi rata-rata satu halaman punya sekitar 19 link keluar hasil OCR, angka yang masuk akal untuk halaman Wikipedia versi sekolah yang dipakai di kompetisi ini.

## Jarak ke target lewat BFS

Untuk satu target, jarak (jumlah klik minimal) dari semua node lain ke target itu dihitung sekali lewat breadth first search di graf yang arahnya dibalik. Jalur target menuju X di graf terbalik sama persis dengan jalur X menuju target di graf asli, jadi triknya cukup menjalankan BFS satu kali dari target di graf terbalik untuk mendapat jarak ke semua node sekaligus. Hasilnya disimpan di cache per target, karena tiap target di train dan test selalu muncul di 25 baris berbeda, cache ini mencegah BFS yang sama dihitung berulang.

In [ ]:
_dist_cache: dict = {}


def dist_to_target(target: int) -> dict:
    if target not in _dist_cache:
        _dist_cache[target] = nx.single_source_shortest_path_length(G_REV, target)
    return _dist_cache[target]

## Aturan prediksi

Untuk satu baris (current, target), kandidat jawabannya adalah semua link hasil OCR dari halaman current. Kalau OCR tidak menemukan link sama sekali di halaman itu (jarang terjadi), semua artikel dipakai sebagai kandidat cadangan. Tiap kandidat diberi skor berupa pasangan (jarak ke target, minus skor hub), lalu dipilih kandidat dengan skor terkecil, artinya jarak paling dekat ke target dulu, kalau ada beberapa kandidat dengan jarak sama baru dipilih yang skor hubnya paling besar. Aturan ini sama sekali tidak melihat next_article_id di data train, jadi hasilnya berlaku sama baiknya untuk target manapun termasuk yang belum pernah muncul di train.

In [ ]:
ALL_NODES = np.arange(N)


def predict_next(current: int, target: int) -> int:
    cands = [cid for cid, _ in OCR_LINKS.get(current, []) if cid != current]
    if not cands:
        cands = [i for i in ALL_NODES if i != current]
    dmap = dist_to_target(target)
    best_c, best_key = None, None
    for c in cands:
        d = dmap.get(c, CFG.UNREACHABLE_DIST)
        key = (d, -HUB_SCORE[c])
        if best_key is None or key < best_key:
            best_key, best_c = key, c
    return int(best_c)


def predict_batch(df: pd.DataFrame) -> np.ndarray:
    return np.array([predict_next(int(c), int(t))
                     for c, t in zip(df.current_article_id, df.target_article_id)])

## Validasi

Tidak ada model yang dilatih di pendekatan ini, predict_next murni fungsi deterministik dari graf ditambah current dan target, jadi tidak ada risiko overfit atau bocor dari train ke test seperti pada pendekatan LightGBM sebelumnya. Karena itu akurasi di seluruh data train sudah jadi estimasi yang wajar untuk performa di test, tanpa perlu skema validasi silang yang rumit.

In [ ]:
_t0 = time.time()
train_pred = predict_batch(train)
train_acc = float((train_pred == train.next_article_id.values).mean())
print(f"Train accuracy (graph-only, full {len(train)} rows): {train_acc*100:.2f}%  "
      f"({time.time()-_t0:.1f}s)")

_hub_baseline = float((train.next_article_id.values == int(np.argmax(HUB_SCORE))).mean())
print(f"Sanity baseline (always predict the single top hub): {_hub_baseline*100:.2f}%")

Akurasi di train tembus 60.40%, jauh di atas baseline naif yang cuma selalu menebak hub terpopuler (11.64%, angka ini juga persis sama dengan porsi baris next yang benar benar artikel paling populer di eksplorasi awal tadi). Ini menunjukkan jarak ke target di graf memberi sinyal yang jauh lebih kuat dibanding sekadar menebak artikel populer, dan pendekatan graf sederhana ini mengalahkan model LightGBM yang sempat dicoba sebelumnya (0.28) dengan selisih besar.

## Prediksi dan submission

In [ ]:
test_pred = predict_batch(test)

FALLBACK_ID = int(np.argmax(HUB_SCORE))
pred_df = pd.DataFrame({"state_id": test.state_id.values,
                        "predicted_next_article_id": test_pred.astype(int)})
submission = sample_sub[["state_id"]].merge(pred_df, on="state_id", how="left")
submission["predicted_next_article_id"] = (
    submission["predicted_next_article_id"].fillna(FALLBACK_ID).astype(int))
submission.to_csv(CFG.SUB_PATH, index=False)

assert len(submission) == len(sample_sub), "submission row count mismatch"
assert submission["predicted_next_article_id"].notna().all(), "missing predictions"
print("Saved:", CFG.SUB_PATH, submission.shape)
print(submission.head(10).to_string(index=False))

## Hasil

Akurasi di data train sekitar 60.4%, jauh di atas baseline hub tunggal (11.6%) maupun pendekatan LightGBM sebelumnya (0.28). Submission berisi 6000 baris sesuai jumlah baris di states_test, urutannya mengikuti sample_submission. Kunci utama pendekatan ini ada di dua hal, graf hyperlink yang direkonstruksi lewat OCR sehingga kandidat jawaban selalu link yang benar benar ada di halaman, dan jarak terpendek ke target yang langsung merepresentasikan strategi navigasi Wikipedia yang sebenarnya tanpa perlu belajar dari label.